# WORKSHOP 1 : BRONZE CUT — GROSS DATA INSTITUTION

**Lakehouse** : `Lakehouse_bronze` · **Notebook** : `NB_01_Bronze`
**Objective**: Ingest raw data **without transformation** (source of immutable truth).

The Bronze layer keeps the data exactly as it happens (null values, error codes, heteroclites formats, duplicates). Cleaning is done only in Silver.

## Prerequisite

1. Workspace (ex. `ws_medallion_electricity`) and Lakehouse **`Lakehouse_bronze`**, **attached to this notebook**
2. In **Files** of Lakehouse, create folder`data_bronze_notebooks`and load the 5 files:

| File | Lines | Columns |
| --- | --- | --- |
| `consumption_raw.csv` | 18 144 | timestamp, site_id, consumption_mw, voltage_v, frequency_hz, status |
| `market_prices.csv` | 2 880 | timestamp, price_eur_mwh, volume_mwh, market |
| `weather_data.csv` | 720 | timestamp, temperature_c, humidity_pct, wind_speed_ms, cloud_cover_pct |
| `sites_reference.csv` | 6 | site_id, site_type, capacity_mw, flexible, baseline_mw, curtailment_price_eur_mwh, region |
| `maintenance_events.csv` | 8 | site_id, start_time, end_time, maintenance_type, impact_pct |

Anomalies of`consumption_raw.csv`multiple timestamp formats, null values, error codes (−888, -999, -777), duplicates.

---
## Cell 1: Spark Configuration and Bronze Schema Creation

Check Spark session and create schema`bronze` (`IF NOT EXISTS` = idempotent).

In [ ]:
# Vérifier que Spark est disponible
print(f"✅ Spark version: {spark.version}")

# Créer le schéma bronze s'il n'existe pas encore
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
print("✅ Schéma bronze prêt")

# Confirmer le lakehouse actif
df_schemas = spark.sql("SHOW SCHEMAS").toPandas()
print(f"✅ Schémas disponibles : {list(df_schemas.iloc[:,0])}")

**Attended**: version of Spark 3.5.x,`Schéma bronze prêt`, and a list of diagrams containing`bronze` (ainsi que `default` / `dbo`created by Fabric).

---
## Cellule 2 : Import `consumption_raw.csv`→ Delta table

`header=true` + `inferSchema=true`, then write Delta in the schema`bronze` en mode `overwrite`. Line counting is a systematic check.

In [ ]:
# Charger le CSV avec PySpark
df_consumption = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/data_bronze_notebooks/consumption_raw.csv")

# Compter les lignes avant écriture
nb_lignes = df_consumption.count()

# Créer la table dans le schéma bronze
df_consumption.write.mode("overwrite").format("delta").saveAsTable("bronze.consumption_raw")

print(f"✅ Table bronze.consumption_raw créée : {nb_lignes} lignes")

**Intended**:`✅ Table bronze.consumption_raw créée : 18144 lignes`

18 144 lines = 17,280 theoretical (6 sites × 4 measurements/h × 24 hours × 30 days) + 864 duplicates.

---
## Cell 3: First view of raw data

Visual inspection: timestamp formats, null values, negative error codes.

In [ ]:
%%sql
SELECT * FROM bronze.consumption_raw LIMIT 20

**To be observed** (the exact values depend on the order of the lines):
- timestamps en **plusieurs formats** : `2025-01-03T04:30:00`, `17/01/2025 18:15:00`, `2025-01-22 01:00:00`
- `consumption_mw`**empty** (NULL) and codes **−888 / -777 / -999**, with`status = ERROR`
- positive values consistent with site capacity (0.6 to 5 MW)

---
## Cell 4: Diagnostic quality`consumption_raw`

Total lines, single lines`(timestamp, site_id)`, duplicates, null, error codes and outputs (> 10 MW).

In [ ]:
%%sql
-- Compter les NULL, codes erreur, doublons
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT timestamp, site_id) as unique_rows,
    COUNT(*) - COUNT(DISTINCT timestamp, site_id) as duplicates,
    SUM(CASE WHEN consumption_mw IS NULL THEN 1 ELSE 0 END) as null_values,
    SUM(CASE WHEN consumption_mw < 0 THEN 1 ELSE 0 END) as error_codes,
    SUM(CASE WHEN consumption_mw > 10 THEN 1 ELSE 0 END) as outliers
FROM bronze.consumption_raw

**Intended**:

| total_rows | unique_rows | duplicates | null_values | error_codes | outliers |
| --- | --- | --- | --- | --- | --- |
| 18144 | 17280 | 864 | 554 | 319 | 27 |

- **864 duplicates** (4.8%) → deduce in Silver
- **554 NULL** (3%) → Filter or impute
- **319 error codes** (1.8%): -888 (sensor failure), -999 (network failure), -777 (calibration)
- **27 outliers** (0.15 %): > 10 MW above the maximum capacity of a site (5 MW)

---
## Cell 5: Import the other four files

In [ ]:
# Market prices
spark.read.csv("Files/data_bronze_notebooks/market_prices.csv", header=True, inferSchema=True) \
    .write.mode("overwrite").format("delta").saveAsTable("bronze.market_prices")

# Weather
spark.read.csv("Files/data_bronze_notebooks/weather_data.csv", header=True, inferSchema=True) \
    .write.mode("overwrite").format("delta").saveAsTable("bronze.weather_data")

# Sites reference
spark.read.csv("Files/data_bronze_notebooks/sites_reference.csv", header=True, inferSchema=True) \
    .write.mode("overwrite").format("delta").saveAsTable("bronze.sites_reference")

# Maintenance events
spark.read.csv("Files/data_bronze_notebooks/maintenance_events.csv", header=True, inferSchema=True) \
    .write.mode("overwrite").format("delta").saveAsTable("bronze.maintenance_events")

print("✅ Toutes les tables Bronze créées")

**Intended**:`✅ Toutes les tables Bronze créées`. In the Lakehouse Explorer, 5 tables under **bronze** (click **Refresh** if necessary).

---
## Cell 6: Verification and summary of all tables

In [ ]:
%%sql
SELECT 'consumption' as table_name, COUNT(*) as rows FROM bronze.consumption_raw
UNION ALL
SELECT 'prices', COUNT(*) FROM bronze.market_prices
UNION ALL
SELECT 'weather', COUNT(*) FROM bronze.weather_data
UNION ALL
SELECT 'sites', COUNT(*) FROM bronze.sites_reference
UNION ALL
SELECT 'maintenance', COUNT(*) FROM bronze.maintenance_events

**Extended** : consumption 18144 · prices 2880 · weather 720 · sites 6 · maintenance 8 (total 21 758).

---
## Cell 7: Synthetic Quality Ratio

The numbers are **calculated from Bronze tables** (and not written in hard form).

In [ ]:
import pandas as pd

# --- Volumétrie
tables = ["consumption_raw", "market_prices", "weather_data", "sites_reference", "maintenance_events"]
n = {t: spark.table(f"bronze.{t}").count() for t in tables}
total = sum(n.values())

# --- Anomalies de consumption_raw
q = spark.sql('''
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT timestamp, site_id) AS duplicates,
    SUM(CASE WHEN consumption_mw IS NULL THEN 1 ELSE 0 END) AS null_values,
    SUM(CASE WHEN consumption_mw < 0 THEN 1 ELSE 0 END) AS error_codes,
    SUM(CASE WHEN consumption_mw > 10 THEN 1 ELSE 0 END) AS outliers
FROM bronze.consumption_raw
''').first()

# Lignes avec au moins une anomalie (doublon = occurrence supplémentaire)
n_anom = spark.sql('''
SELECT COUNT(*) FROM (
    SELECT consumption_mw,
           ROW_NUMBER() OVER (PARTITION BY timestamp, site_id ORDER BY timestamp) AS rn
    FROM bronze.consumption_raw
) WHERE rn > 1 OR consumption_mw IS NULL OR consumption_mw < 0 OR consumption_mw > 10
''').first()[0]

tot = q["total_rows"]
pct = lambda x: f"{100 * x / tot:.1f}%"

rapport_bronze = f'''
╔════════════════════════════════════════════════════════════════════╗
║                  RAPPORT QUALITÉ COUCHE BRONZE                     ║
║                   Pipeline Medallion - Électricité                 ║
╚════════════════════════════════════════════════════════════════════╝

📊 RÉSUMÉ DE L'INGESTION
✅ {len(tables)} fichiers ingérés, {len(tables)} tables Delta dans le schéma 'bronze'
✅ {total:,} lignes totales stockées

📋 TABLES CRÉÉES
• bronze.consumption_raw      : {n['consumption_raw']:>6,} lignes (données IoT)
• bronze.market_prices        : {n['market_prices']:>6,} lignes (prix spot EPEX)
• bronze.weather_data         : {n['weather_data']:>6,} lignes (conditions météo)
• bronze.sites_reference      : {n['sites_reference']:>6,} lignes (données maîtres sites)
• bronze.maintenance_events   : {n['maintenance_events']:>6,} lignes (événements maintenance)

⚠️ ANOMALIES DÉTECTÉES DANS bronze.consumption_raw
🔴 DOUBLONS     : {q['duplicates']} lignes ({pct(q['duplicates'])})  → Silver : dédupliquer
🔴 VALEURS NULL : {q['null_values']} lignes ({pct(q['null_values'])})  → Silver : filtrer ou imputer
🔴 CODES ERREUR : {q['error_codes']} lignes ({pct(q['error_codes'])})  → -888 défaut capteur, -999 panne réseau, -777 calibrage
🔴 OUTLIERS     : {q['outliers']} lignes ({pct(q['outliers'])})  → consommation > 10 MW (capacité max 5 MW)
🟡 TIMESTAMPS HÉTÉROGÈNES : 3 formats
   ISO 8601 (2025-01-03T04:30:00) · Français (17/01/2025 18:15:00) · Espace (2025-01-22 01:00:00)

📈 TAUX DE QUALITÉ GLOBAL
Lignes sans anomalie identifiée   : {tot - n_anom:,} / {tot:,} = {pct(tot - n_anom)} ✅
Lignes avec au moins une anomalie : {n_anom:,} / {tot:,} = {pct(n_anom)} ⚠️

🎯 PRIORITÉS SILVER
1. Normaliser les timestamps (ISO 8601)
2. Dédupliquer ({q['duplicates']} doublons)
3. Traiter les codes erreur (-888, -999, -777)
4. Gérer les NULL (interpolation ou exclusion)
5. Valider les outliers ({q['outliers']} mesures > 10 MW)
6. Enrichir avec sites_reference, weather, maintenance

SOURCE DE VÉRITÉ : immutable — aucune transformation appliquée en Bronze.

Rapport généré : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
'''

print(rapport_bronze)

**Intended**: 21,758 lines; 864 (4.8 %), NULL 554 (3.1%), error codes 319 (1.8%), other 27 (0.1%); **1 718 lines** with at least one anomaly (9.5%) or **16 426 lines (90.5%)** without anomaly.

---
## Executive summary

- **5 Delta tables** in the diagram`bronze`of the`Lakehouse_bronze`, **21 758 lines**, no transformation
- **Idempotence** : `CREATE SCHEMA IF NOT EXISTS` + mode `overwrite`→ Notebook can be restarted without error or duplicate
- **Diagnosis**: duplicates, null, error codes, output and timestamp formats documented and quantified
- **Continued**: Silver Notebook will read these tables`bronze.*`to produce clean tables in the schema`silver`